 # Imports, Primitives

In [ ]:
from google.colab import drive
import os
import time
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses, regularizers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from PIL import Image, ImageFont
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

drive.mount('/content/drive')

ZIP_SRC  = "/content/drive/MyDrive/Deep Learning Project/GoogleStreetViewImagesNew.zip"
LOCAL_DIR = "/content/streetview_dataset"

if not os.path.exists(LOCAL_DIR):
    t0 = time.time()
    !cp "{ZIP_SRC}" /content/streetview_dataset.zip
    !unzip -q /content/streetview_dataset.zip -d {LOCAL_DIR}
    print(f"Staged in {time.time() - t0:.1f}s")

!ls {LOCAL_DIR} | head

Mounted at /content/drive
Staged in 63.8s
GoogleStreetViewImagesNew
__MACOSX


# EfficientNetV2S Backbone Early Fusion (Compass Tiling)

The four cardinal heading images are spatially tiled into a 2x2 compass grid before being passed through a single EfficientNetV2S backbone, enabling the model to learn cross-view spatial relationships in a single forward pass.

## Architecture

    Input (4 × H × W × C)              ← 4 cardinal views per location
            ↓
    TimeDistributed(Augmentation)      ← per-view random transforms (train only)
            ↓
    Compass Tile Assembly              ← stacks views into a 2×2 grid
         [ N | E ]
         [ W | S ]  →  (batch, 768, 768, 3)
            ↓
    Resize to (384, 384, 3)
            ↓
    EfficientNetV2S Backbone           ← single forward pass, sees all 4 headings
            ↓
    GlobalAveragePooling2D
            ↓
    Dense(512, gelu) → Dropout → Dense(3, softmax)

In [ ]:
"""
Early fusion via 2x2 compass tiling.

"""

import tensorflow as tf
import numpy as np
import pandas as pd
import os
import math

IMG_SIZE   = 384
BATCH_SIZE = 16
EPOCHS_P1  = 20
EPOCHS_P2  = 30
CSV_PATH   = "/content/drive/MyDrive/Deep Learning Project/images_bra-3.csv"
PREPROCESS = tf.keras.applications.efficientnet_v2.preprocess_input

IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

def dataloading_multiview(csv_path, img_size=384, batch_size=16, preprocess_fn=None):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["income_group"]).reset_index(drop=True)
    df["full_path"] = df["file_path"].apply(
        lambda f: os.path.join("/content/streetview_dataset/", f)
    )
    df["loc_id"] = df["file_path"].apply(
        lambda f: os.path.basename(f).split("_h")[0]
    )
    df["heading"] = df["file_path"].apply(
        lambda f: int(os.path.basename(f).split("_h")[1].replace(".jpg", ""))
    )
    df = df[df["full_path"].apply(os.path.exists)].reset_index(drop=True)

    grouped = (
        df.sort_values("heading")
          .groupby("loc_id")
          .agg(paths=("full_path", list), label=("income_group", "first"))
          .reset_index()
    )
    grouped = grouped[grouped["paths"].apply(len) == 4].reset_index(drop=True)
    print(f"Locations with all 4 headings: {len(grouped)}")

    np.random.seed(50)
    loc_ids = grouped["loc_id"].values.copy()
    np.random.shuffle(loc_ids)
    n = len(loc_ids)
    train_ids = set(loc_ids[:int(0.70 * n)])
    val_ids   = set(loc_ids[int(0.70 * n):int(0.85 * n)])
    test_ids  = set(loc_ids[int(0.85 * n):])

    train_df = grouped[grouped["loc_id"].isin(train_ids)].reset_index(drop=True)
    val_df   = grouped[grouped["loc_id"].isin(val_ids)].reset_index(drop=True)
    test_df  = grouped[grouped["loc_id"].isin(test_ids)].reset_index(drop=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} locations")

    def make_dataset(split_df, shuffle=False):
        paths_list = split_df["paths"].tolist()
        labels     = split_df["label"].values.astype(np.int32)

        def generator():
            for paths, label in zip(paths_list, labels):
                yield paths, label

        ds = tf.data.Dataset.from_generator(
            generator,
            output_signature=(
                tf.TensorSpec(shape=(4,), dtype=tf.string),
                tf.TensorSpec(shape=(),   dtype=tf.int32)
            )
        )

        def load_fn(paths, label):

            imgs = tf.map_fn(
                lambda p: tf.cast(
                    tf.image.resize(
                        tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                        [img_size, img_size]
                    ), tf.float32
                ),
                paths,
                fn_output_signature=tf.TensorSpec(
                    shape=(img_size, img_size, 3), dtype=tf.float32
                )
            )
            return imgs, label

        if shuffle:
            ds = ds.shuffle(buffer_size=len(split_df), seed=42)
        ds = (ds
              .map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(batch_size)
              .repeat()
              .prefetch(tf.data.AUTOTUNE))
        return ds

    train_ds = make_dataset(train_df, shuffle=True)
    val_ds   = make_dataset(val_df)
    test_ds  = make_dataset(test_df)

    steps = {
        "train": math.ceil(len(train_df) / batch_size),
        "val":   math.ceil(len(val_df)   / batch_size),
        "test":  math.ceil(len(test_df)  / batch_size),
    }
    print(f"Steps — train: {steps['train']} | val: {steps['val']} | test: {steps['test']}")

    return train_ds, val_ds, test_ds, steps


# COMPASS TILING
def assemble_compass_tile(images):
    """
    images: (batch, 4, H, W, 3) ordered [N=0°, E=90°, S=180°, W=270°]
    returns: (batch, 2H, 2W, 3) arranged as
        [ N | E ]
        [ W | S ]
    """
    n = images[:, 0]   # 0°
    e = images[:, 1]   # 90°
    s = images[:, 2]   # 180°
    w = images[:, 3]   # 270°
    top    = tf.concat([n, e], axis=2)   # concat along width
    bottom = tf.concat([w, s], axis=2)
    return tf.concat([top, bottom], axis=1)   # concat along height


# ── PER-HEADING AUGMENTATION
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")


# MODEL
def build_model():
    base_model = tf.keras.applications.EfficientNetV2S(
        input_shape=IMG_SHAPE, include_top=False, weights="imagenet"
    )
    base_model.trainable = False

    inputs = tf.keras.Input(
        shape=(4, IMG_SIZE, IMG_SIZE, 3), name="multiview_input"
    )

    # Per-heading augmentation
    x = tf.keras.layers.TimeDistributed(
            data_augmentation, name="td_aug")(inputs)        # (b, 4, 384, 384, 3)

    # Assemble compass tile -> (b, 768, 768, 3)
    x = tf.keras.layers.Lambda(
            assemble_compass_tile,
            output_shape=(2*IMG_SIZE, 2*IMG_SIZE, 3),
            name="compass_tile")(x)

    # Resize back to backbone input size -> (b, 384, 384, 3)
    x = tf.keras.layers.Resizing(IMG_SIZE, IMG_SIZE, name="resize_to_backbone")(x)

    # Apply backbone preprocessing (expects values in raw [0, 255] from JPEG decode)
    x = tf.keras.layers.Lambda(
            PREPROCESS,
            output_shape=(IMG_SIZE, IMG_SIZE, 3),
            name="preprocess")(x)

    # Single-pass through the backbone — this is where cross-heading features form
    x = base_model(x, training=False)                        # (b, 12, 12, 1280)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x) # (b, 1280)

    # Head — identical to your other variants for clean comparison
    x = tf.keras.layers.Dropout(0.4, name="drop_1")(x)
    x = tf.keras.layers.Dense(
            512, activation="gelu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name="dense_hidden")(x)
    x = tf.keras.layers.Dropout(0.3, name="drop_2")(x)
    outputs = tf.keras.layers.Dense(
            3, activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name="predictions")(x)

    model = tf.keras.Model(
        inputs, outputs, name="EfficientNetV2S_EarlyFusion_Tiling"
    )
    return model, base_model


# ── MAIN
train_ds, val_ds, test_ds, steps = dataloading_multiview(
    csv_path=CSV_PATH,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    preprocess_fn=PREPROCESS,
)

# Sanity check shapes
image_batch, label_batch = next(iter(train_ds))
print("Image batch shape :", image_batch.shape)   # (16, 4, 384, 384, 3)
print("Label batch shape :", label_batch.shape)   # (16,)

model, base_model = build_model()
model.summary()

ckpt = "/content/drive/MyDrive/Deep Learning Project/best_early_fusion_tiling.keras"

# Phase 1: frozen backbone
print("\n=== Phase 1: Feature Extraction (frozen backbone) ===")

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt, monitor="val_accuracy",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2,
        patience=4, min_lr=1e-6, verbose=1
    ),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1,
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    callbacks=callbacks_p1, verbose=1
)

print("\n=== Test Evaluation (Phase 1) ===")
loss_p1, acc_p1 = model.evaluate(test_ds, steps=steps["test"], verbose=1)
print(f"Phase 1 test  →  acc: {acc_p1:.4f}  loss: {loss_p1:.4f}")

p1_weights = "/content/drive/MyDrive/Deep Learning Project/p1_early_fusion.weights.h5"
model.save_weights(p1_weights)

# Phase 2: fine-tune top 30% of backbone (matched to LSTM run)
print("\n=== Phase 2: Fine-Tuning (top 30% of backbone) ===")

base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.70)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Freezing layers 0–{fine_tune_at} / {len(base_model.layers)}")
print(f"Trainable backbone layers: {trainable} / {len(base_model.layers)}")

cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=EPOCHS_P2 * steps["train"],
    alpha=1e-7,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_lr),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt, monitor="val_accuracy",
        save_best_only=True, verbose=1
    ),
]

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2,
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    callbacks=callbacks_p2, verbose=1
)

print("\n=== Test Evaluation (Phase 2) ===")
loss_p2, acc_p2 = model.evaluate(test_ds, steps=steps["test"], verbose=1)
print(f"Phase 2 test  →  acc: {acc_p2:.4f}  loss: {loss_p2:.4f}")

if acc_p2 < acc_p1:
    print(f"Phase 2 degraded ({acc_p1:.4f} → {acc_p2:.4f}), restoring Phase 1")
    model.load_weights(p1_weights)
    acc_p2 = acc_p1

print("\n" + "=" * 58)
print("   EARLY FUSION (2x2 COMPASS TILING) RESULTS")
print("=" * 58)
print(f"Phase 1 test acc          : {acc_p1*100:.2f}%")
print(f"Phase 2 test acc          : {acc_p2*100:.2f}%")
print(f"")
print(f"Reference benchmarks:")
print(f"  Single-image baseline   : 66.35%")
print(f"  Post-hoc median         : 74.06%")
print(f"  LSTM late fusion        : 74.45%")
print(f"  Self-attention late fus.: ~63%")
print(f"  Best fusion variant(avg): 64.72%")
print("=" * 58)

Locations with all 4 headings: 5476
Train: 3833 | Val: 821 | Test: 822 locations
Steps — train: 240 | val: 52 | test: 52
Image batch shape : (16, 4, 384, 384, 3)
Label batch shape : (16,)
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "EfficientNetV2S_EarlyFusion_Tiling"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ multiview_input (InputLayer)    │ (None, 4, 384, 384, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_aug (TimeDistributed)        │ (None, 4, 384, 384, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ compass_tile (Lambda)           │ (None, 768, 768, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resize_to_backbone (Resizing)   │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ preprocess (Lambda)             │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 12, 12, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,988,771 (80.07 MB)

 Trainable params: 657,411 (2.51 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


=== Phase 1: Feature Extraction (frozen backbone) ===
Epoch 1/20
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4226 - loss: 1.2098
Epoch 1: val_accuracy improved from None to 0.55298, saving model to /content/drive/MyDrive/Deep Learning Project/best_early_fusion_tiling.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Deep Learning Project/best_early_fusion_tiling.keras
240/240 ━━━━━━━━━━━━━━━━━━━━ 55s 113ms/step - accuracy: 0.4490 - loss: 1.1501 - val_accuracy: 0.5530 - val_loss: 1.0606 - learning_rate: 0.0010
Epoch 2/20
239/240 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.4935 - loss: 1.0734
Epoch 2: val_accuracy did not improve from 0.55298
240/240 ━━━━━━━━━━━━━━━━━━━━ 18s 77ms/step - accuracy: 0.5030 - loss: 1.0691 - val_accuracy: 0.5323 - val_loss: 1.0408 - learning_rate: 0.0010
Epoch 3/20
239/240 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5151 - loss: 1.0608
Epoch 3: val_accuracy improved from 0.55298 to 0.56516, saving model to /content/drive/MyD